# Лабораторная работа №6

## Предобработка и классификация текста

**Цель лабораторной работы:** изучение методов предобработки и классификации текстов.

**Задание:**

### Часть 1

Для произвольного предложения или текста выполнить:

1. Токенизацию.
2. Частеречную разметку.
3. Лемматизацию.
4. Выделение именованных сущностей.
5. Разбор предложения.

### Часть 2

Для произвольного набора данных, предназначенного для классификации текстов, решить задачу классификации текста двумя способами:

1. На основе `CountVectorizer` или `TfidfVectorizer`.
2. На основе модели `Word2Vec`, `GloVe` или `fastText`.

После этого необходимо сравнить качество полученных моделей.


## 1. Установка и импорт библиотек

В работе используются библиотеки:

- `spaCy` — для предобработки текста;
- `scikit-learn` — для классификации и оценки качества;
- `gensim` — для обучения модели Word2Vec;
- `pandas`, `numpy`, `matplotlib` — для анализа данных и визуализации.

Если нужные библиотеки отсутствуют, они будут установлены автоматически.


In [ ]:
import sys
import subprocess
import importlib.util

packages = ["spacy", "gensim", "sklearn", "pandas", "numpy", "matplotlib"]

install_map = {
    "sklearn": "scikit-learn"
}

for package in packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            install_map.get(package, package)
        ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spacy
from spacy import displacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from gensim.models import Word2Vec

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

## 2. Загрузка языковой модели spaCy

Для выполнения токенизации, лемматизации, частеречной разметки, выделения сущностей и синтаксического разбора используется модель `en_core_web_sm`.

Если модель не установлена, она будет загружена автоматически.

В данной работе используется английский текст, так как для него доступны стабильные готовые модели spaCy.


In [ ]:
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "spacy",
        "download",
        "en_core_web_sm"
    ])
    nlp = spacy.load("en_core_web_sm")

print("Модель spaCy успешно загружена.")

# Часть 1. Предобработка текста

В первой части лабораторной работы выполняется обработка одного предложения.

Будут выполнены:

- токенизация;
- частеречная разметка;
- лемматизация;
- выделение именованных сущностей;
- синтаксический разбор предложения.


## 3. Исходный текст

Для анализа выбран текст, содержащий имена собственные, название организации, дату и место. Это позволяет продемонстрировать работу распознавания именованных сущностей.


In [ ]:
text = "In 2024, Elon Musk announced that Tesla would open a new research center in Berlin."

doc = nlp(text)

print("Исходный текст:")
print(text)

## 4. Токенизация

**Токенизация** — это процесс разбиения текста на отдельные элементы: слова, числа, знаки препинания и другие токены.

Токенизация является базовым этапом обработки естественного языка.


In [ ]:
tokens = [token.text for token in doc]

print("Токены:")
print(tokens)

tokens_df = pd.DataFrame({
    "№": range(1, len(tokens) + 1),
    "Токен": tokens
})

display(tokens_df)

## 5. Частеречная разметка

**Частеречная разметка** — это определение части речи для каждого токена.

Например:

- существительное;
- глагол;
- прилагательное;
- имя собственное;
- предлог;
- знак препинания.

Частеречная разметка используется для дальнейшего анализа структуры текста.


In [ ]:
pos_df = pd.DataFrame([
    {
        "Токен": token.text,
        "Часть речи": token.pos_,
        "Подробный тег": token.tag_,
        "Описание тега": spacy.explain(token.tag_)
    }
    for token in doc
])

display(pos_df)

## 6. Лемматизация

**Лемматизация** — это приведение слова к его начальной форме.

Например:

- `announced` → `announce`;
- `would` → `would`;
- `centers` → `center`.

Лемматизация позволяет уменьшить количество различных форм одного и того же слова.


In [ ]:
lemma_df = pd.DataFrame([
    {
        "Токен": token.text,
        "Лемма": token.lemma_,
        "Часть речи": token.pos_
    }
    for token in doc
])

display(lemma_df)

## 7. Выделение именованных сущностей

**Named Entity Recognition**, или распознавание именованных сущностей, позволяет находить в тексте:

- имена людей;
- организации;
- географические объекты;
- даты;
- денежные суммы;
- и другие значимые объекты.

В данном примере можно выделить персону, организацию, город и дату.


In [ ]:
entities = [
    {
        "Сущность": ent.text,
        "Тип": ent.label_,
        "Описание": spacy.explain(ent.label_)
    }
    for ent in doc.ents
]

entities_df = pd.DataFrame(entities)

display(entities_df)

## 8. Синтаксический разбор предложения

**Синтаксический разбор** показывает грамматические связи между словами в предложении.

Для каждого токена определяются:

- родительское слово;
- тип синтаксической зависимости;
- описание зависимости.

Например, можно определить подлежащее, сказуемое, дополнение и обстоятельства.


In [ ]:
parse_df = pd.DataFrame([
    {
        "Токен": token.text,
        "Лемма": token.lemma_,
        "Часть речи": token.pos_,
        "Зависимость": token.dep_,
        "Описание зависимости": spacy.explain(token.dep_),
        "Родитель": token.head.text
    }
    for token in doc
])

display(parse_df)

## 9. Визуализация синтаксического разбора

В этом блоке выполняется визуализация дерева зависимостей предложения.

Если ноутбук открыт в Jupyter Notebook, схема будет отображена прямо под ячейкой.


In [ ]:
displacy.render(doc, style="dep", jupyter=True)

## 10. Визуализация именованных сущностей

В этом блоке выполняется визуализация найденных именованных сущностей в тексте.


In [ ]:
displacy.render(doc, style="ent", jupyter=True)

# Часть 2. Классификация текста

Во второй части лабораторной работы решается задача классификации текстов.

Будет создан небольшой искусственный датасет из трех классов:

1. `technology` — тексты о технологиях.
2. `sports` — тексты о спорте.
3. `medicine` — тексты о медицине.

Классификация будет выполнена двумя способами:

1. TF-IDF + Logistic Regression.
2. Word2Vec + Logistic Regression.


## 11. Создание текстового датасета

Так как по заданию разрешен произвольный набор данных, в работе используется искусственно сгенерированный датасет.

Каждый текст относится к одному из трех классов.


In [ ]:
technology_texts = [
    "Artificial intelligence improves software development and data analysis",
    "New smartphone processors provide better performance and battery life",
    "Cloud computing allows companies to store data on remote servers",
    "Cybersecurity protects computer systems from digital attacks",
    "Machine learning models are trained on large datasets",
    "The new operating system update improves device security",
    "Developers use Python for automation and backend services",
    "Neural networks are widely used in image recognition",
    "Databases store structured information for business applications",
    "Robots can automate production and logistics processes"
]

sports_texts = [
    "The football team won the final match after extra time",
    "Basketball players trained hard before the championship",
    "The tennis tournament attracted many professional athletes",
    "Running every morning improves endurance and physical health",
    "The coach changed the strategy during the second half",
    "The hockey club signed a new goalkeeper",
    "Swimming competitions require strength and technique",
    "The athlete broke the national record in sprint",
    "Fans supported their team at the stadium",
    "Cycling is popular among people who enjoy outdoor sports"
]

medicine_texts = [
    "Doctors recommend regular exercise and healthy nutrition",
    "The hospital introduced a new method of patient treatment",
    "Vaccination helps protect people from dangerous diseases",
    "Medical research improves diagnosis and therapy",
    "A balanced diet reduces the risk of heart disease",
    "Nurses monitor patients after surgery",
    "The new medicine was tested in clinical trials",
    "Early diagnosis increases the effectiveness of treatment",
    "Physical therapy helps patients recover after injuries",
    "Public health programs focus on disease prevention"
]

texts = []
labels = []

for text in technology_texts:
    texts.append(text)
    labels.append("technology")

for text in sports_texts:
    texts.append(text)
    labels.append("sports")

for text in medicine_texts:
    texts.append(text)
    labels.append("medicine")

# Увеличиваем датасет за счет небольших вариантов предложений
augmented_texts = []
augmented_labels = []

for text, label in zip(texts, labels):
    augmented_texts.append(text)
    augmented_labels.append(label)

    augmented_texts.append(text.lower())
    augmented_labels.append(label)

    augmented_texts.append(text + " today")
    augmented_labels.append(label)

data = pd.DataFrame({
    "text": augmented_texts,
    "label": augmented_labels
})

display(data.head(10))

print("Размер датасета:", data.shape)
print("\nРаспределение классов:")
print(data["label"].value_counts())

## 12. Разделение данных на обучающую и тестовую выборки

Для оценки качества модели датасет делится на две части:

- обучающая выборка — используется для обучения модели;
- тестовая выборка — используется для проверки качества на новых данных.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["text"],
    data["label"],
    test_size=0.3,
    random_state=42,
    stratify=data["label"]
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

## 13. Способ 1. Классификация на основе TF-IDF

**TF-IDF** преобразует текст в числовой вектор.

Он учитывает:

- частоту слова в документе;
- важность слова относительно всего корпуса документов.

После векторизации используется модель логистической регрессии.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("Accuracy TF-IDF модели:", accuracy_tfidf)
print("\nОтчет классификации:")
print(classification_report(y_test, y_pred_tfidf))

## 14. Матрица ошибок для TF-IDF модели

Матрица ошибок показывает, какие классы модель определяет правильно, а какие путает между собой.


In [ ]:
cm_tfidf = confusion_matrix(y_test, y_pred_tfidf, labels=tfidf_model.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_tfidf,
    display_labels=tfidf_model.classes_
)

disp.plot()
plt.title("Матрица ошибок: TF-IDF + Logistic Regression")
plt.show()

## 15. Подготовка данных для Word2Vec

**Word2Vec** обучает векторные представления слов.

В отличие от TF-IDF, Word2Vec учитывает контекст слов и позволяет получать плотные векторные представления.

Для классификации текста каждый документ будет представлен как среднее значение векторов его слов.


In [ ]:
def simple_tokenize(text):
    doc = nlp(text.lower())
    return [
        token.lemma_
        for token in doc
        if token.is_alpha and not token.is_stop
    ]


tokenized_texts = [simple_tokenize(text) for text in data["text"]]

print("Пример токенизированного текста:")
print(data["text"].iloc[0])
print(tokenized_texts[0])

## 16. Обучение модели Word2Vec

Модель Word2Vec обучается на текстах из созданного датасета.

Используются следующие параметры:

- `vector_size=50` — размерность вектора слова;
- `window=3` — размер контекстного окна;
- `min_count=1` — учитывать все слова;
- `sg=1` — использовать Skip-Gram архитектуру.


In [ ]:
word2vec_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=50,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    seed=42,
    epochs=100
)

print("Word2Vec модель обучена.")
print("Размер словаря:", len(word2vec_model.wv.index_to_key))
print("Примеры слов в словаре:")
print(word2vec_model.wv.index_to_key[:20])

## 17. Преобразование текстов в векторы Word2Vec

Для классификации текста необходимо получить вектор всего документа.

В данной работе вектор документа вычисляется как среднее арифметическое векторов всех слов документа.


In [ ]:
def text_to_vector(text, model, vector_size=50):
    tokens = simple_tokenize(text)
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)


X_vectors = np.vstack([
    text_to_vector(text, word2vec_model, vector_size=50)
    for text in data["text"]
])

print("Размер матрицы признаков Word2Vec:")
print(X_vectors.shape)

## 18. Способ 2. Классификация на основе Word2Vec

После преобразования текстов в векторы обучается логистическая регрессия.

Таким образом, классификация выполняется не по частотам слов, а по векторным представлениям текстов.


In [ ]:
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_vectors,
    data["label"],
    test_size=0.3,
    random_state=42,
    stratify=data["label"]
)

w2v_model = LogisticRegression(max_iter=1000)
w2v_model.fit(X_train_w2v, y_train_w2v)

y_pred_w2v = w2v_model.predict(X_test_w2v)

accuracy_w2v = accuracy_score(y_test_w2v, y_pred_w2v)

print("Accuracy Word2Vec модели:", accuracy_w2v)
print("\nОтчет классификации:")
print(classification_report(y_test_w2v, y_pred_w2v))

## 19. Матрица ошибок для Word2Vec модели

Матрица ошибок позволяет увидеть, насколько хорошо классификатор на основе Word2Vec различает классы текстов.


In [ ]:
cm_w2v = confusion_matrix(y_test_w2v, y_pred_w2v, labels=w2v_model.classes_)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_w2v,
    display_labels=w2v_model.classes_
)

disp.plot()
plt.title("Матрица ошибок: Word2Vec + Logistic Regression")
plt.show()

## 20. Сравнение качества моделей

В данном блоке сравниваются две модели:

1. `TF-IDF + Logistic Regression`.
2. `Word2Vec + Logistic Regression`.

Сравнение выполняется по метрике Accuracy.


In [ ]:
comparison_df = pd.DataFrame({
    "Модель": [
        "TF-IDF + Logistic Regression",
        "Word2Vec + Logistic Regression"
    ],
    "Accuracy": [
        accuracy_tfidf,
        accuracy_w2v
    ]
})

display(comparison_df)

## 21. График сравнения качества моделей

На графике показано сравнение качества классификации по метрике Accuracy.

Чем выше значение Accuracy, тем лучше модель справилась с классификацией текстов.


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison_df["Модель"], comparison_df["Accuracy"])
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Сравнение качества моделей классификации текста")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y")
plt.show()

## 22. Пример классификации нового текста

В этом блоке проверяется работа обеих моделей на новом тексте, которого не было в обучающей выборке.


In [ ]:
new_texts = [
    "New neural network technology improves data processing",
    "The football player scored a goal in the final game",
    "Doctors tested a new treatment for heart disease"
]

new_tfidf = tfidf_vectorizer.transform(new_texts)
tfidf_predictions = tfidf_model.predict(new_tfidf)

new_w2v = np.vstack([
    text_to_vector(text, word2vec_model, vector_size=50)
    for text in new_texts
])
w2v_predictions = w2v_model.predict(new_w2v)

predictions_df = pd.DataFrame({
    "Текст": new_texts,
    "TF-IDF prediction": tfidf_predictions,
    "Word2Vec prediction": w2v_predictions
})

display(predictions_df)

## 23. Итоговый вывод

В ходе лабораторной работы были изучены методы предобработки и классификации текстов.

В первой части были выполнены:

1. Токенизация.
2. Частеречная разметка.
3. Лемматизация.
4. Распознавание именованных сущностей.
5. Синтаксический разбор предложения.

Во второй части была решена задача классификации текстов двумя способами:

1. На основе TF-IDF и логистической регрессии.
2. На основе Word2Vec и логистической регрессии.

По результатам сравнения можно сделать вывод, что TF-IDF хорошо работает на небольших тематических наборах данных, так как эффективно учитывает ключевые слова классов.

Word2Vec позволяет использовать смысловые векторные представления слов, однако на небольшом датасете его качество может быть ниже, так как для хорошего обучения эмбеддингов требуется больше текстов.


In [ ]:
print("Лабораторная работа №6 выполнена.")
print("Выполнена предобработка текста.")
print("Реализована классификация текста двумя способами: TF-IDF и Word2Vec.")
print("Проведено сравнение качества моделей.")